# bpa_scan

Runs the Best Practice Analyzer (semantic-link-labs `run_model_bpa`) over semantic models and appends one row per rule violation to `BpaFindings` in `ci-eventhouse`.

- **Scope:** `WORKSPACE_IDS` lists the workspaces to scan. Leave it empty to scan every workspace the run identity can see.
- **Read only:** it reads each model's metadata over the XMLA endpoint and writes only to `BpaFindings`. It never changes a model.
- **Access:** the run identity needs Build or higher on each model, and the capacity's XMLA endpoint must allow at least Read. Models it cannot read are skipped and listed at the end.
- **History:** every run appends with a new `ScannedAt`, so the table keeps scan history (30-day retention).

In [ ]:
%pip install -q semantic-link-labs==0.18.1

In [ ]:
# Parameters (override from the schedule or a pipeline if needed)
CLUSTER = "https://trd-3p56kn96phc55ysd7n.z5.kusto.fabric.microsoft.com"
DATABASE = "ci-eventhouse"
WORKSPACE_IDS = ["415f790e-a590-4b59-b30a-c3a70a48de1e"]  # [] = every workspace I can see
MAX_MODELS = 100          # safety cap per run
INGEST_CHUNK = 200        # rows per management command

In [ ]:
import base64, datetime, json
import pandas as pd
import requests
import sempy.fabric as fabric
import sempy_labs as labs

scanned_at = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

if WORKSPACE_IDS:
    workspaces = [(w, None) for w in WORKSPACE_IDS]
else:
    wl = fabric.list_workspaces()
    workspaces = list(zip(wl["Id"], wl["Name"]))

SEVERITY = {"❌": "Error", "⚠": "Warning", "ℹ": "Info"}   # run_model_bpa returns icons

def severity(v):
    s = str(v)
    for icon, name in SEVERITY.items():
        if icon in s:
            return name
    return s

rows, skipped, scanned = [], [], 0
for ws_id, ws_name in workspaces:
    try:
        models = fabric.list_datasets(workspace=ws_id, mode="rest")
    except Exception as e:
        skipped.append((ws_id, None, f"list failed: {str(e)[:120]}"))
        continue
    if ws_name is None:
        ws_name = fabric.resolve_workspace_name(ws_id)
    for _, m in models.iterrows():
        if scanned >= MAX_MODELS:
            break
        scanned += 1
        try:
            df = labs.run_model_bpa(dataset=m["Dataset Id"], workspace=ws_id, return_dataframe=True)
        except Exception as e:
            skipped.append((ws_name, m["Dataset Name"], str(e)[:120]))
            continue
        if df is None or len(df) == 0:
            continue
        for _, r in df.iterrows():
            rows.append({
                "ScannedAt": scanned_at, "WorkspaceId": str(ws_id), "WorkspaceName": ws_name,
                "ModelId": str(m["Dataset Id"]), "ModelName": m["Dataset Name"],
                "Rule": r["Rule Name"], "Category": r["Category"], "Severity": severity(r["Severity"]),
                "Description": f'{r["Object Type"]}: {r["Object Name"]} - {r["Description"]}',
            })

print(f"scanned {scanned} model(s): {len(rows)} finding(s), {len(skipped)} skipped")
for s in skipped:
    print("  skipped:", s)

In [ ]:
# Append to BpaFindings. Rows go in as base64 JSON so no quoting is needed in KQL.
token = notebookutils.credentials.getToken(CLUSTER)
COLS = ["ScannedAt", "WorkspaceId", "WorkspaceName", "ModelId", "ModelName", "Rule", "Category", "Severity", "Description"]

def append(chunk):
    b64 = base64.b64encode(json.dumps(chunk).encode("utf-8")).decode()
    csl = f"""
.set-or-append BpaFindings <|
print j = parse_json(base64_decode_tostring('{b64}'))
| mv-expand r = j
| project ScannedAt = todatetime(r.ScannedAt), WorkspaceId = tostring(r.WorkspaceId), WorkspaceName = tostring(r.WorkspaceName),
          ModelId = tostring(r.ModelId), ModelName = tostring(r.ModelName), Rule = tostring(r.Rule),
          Category = tostring(r.Category), Severity = tostring(r.Severity), Description = tostring(r.Description)
"""
    resp = requests.post(f"{CLUSTER}/v1/rest/mgmt", json={"db": DATABASE, "csl": csl},
                         headers={"Authorization": f"Bearer {token}"}, timeout=120)
    if not resp.ok:
        raise RuntimeError(f"ingest failed {resp.status_code}: {resp.text[:400]}")

for i in range(0, len(rows), INGEST_CHUNK):
    append(rows[i:i + INGEST_CHUNK])
print(f"BpaFindings: appended {len(rows)} row(s) at {scanned_at}")